# Assignment 16 — Chinook Database SQL Questions

Run the notebook in the same folder as `chinook.db`. The queries use SQLite and pandas; charts use matplotlib.

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

conn = sqlite3.connect('chinook.db')
print('Connected to Chinook database')


## Q1. Top 10 genres with the highest number of tracks (Bar Plot)


In [ ]:
sql = '''
SELECT g.name AS genre, COUNT(t.track_id) AS track_count
FROM genre g
JOIN track t ON t.genre_id = g.genre_id
GROUP BY g.genre_id, g.name
ORDER BY track_count DESC
LIMIT 10;
'''
df1 = pd.read_sql_query(sql, conn)
display(df1)
df1.plot(kind='bar', x='genre', y='track_count', legend=False, title='Top 10 Genres by Number of Tracks')
plt.xlabel('Genre')
plt.ylabel('Number of Tracks')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Q2. Top 10 albums that contain the most tracks


In [ ]:
sql = '''
SELECT al.title AS album, ar.name AS artist, COUNT(t.track_id) AS track_count
FROM album al
JOIN artist ar ON ar.artist_id = al.artist_id
JOIN track t ON t.album_id = al.album_id
GROUP BY al.album_id, al.title, ar.name
ORDER BY track_count DESC, al.title
LIMIT 10;
'''
df2 = pd.read_sql_query(sql, conn)
display(df2)

## Q3. Top 10 customers who spent the most money


In [ ]:
sql = '''
SELECT c.customer_id,
       c.first_name || ' ' || c.last_name AS customer,
       ROUND(SUM(i.total), 2) AS total_spent
FROM customer c
JOIN invoice i ON i.customer_id = c.customer_id
GROUP BY c.customer_id, customer
ORDER BY total_spent DESC
LIMIT 10;
'''
df3 = pd.read_sql_query(sql, conn)
display(df3)

## Q4. Top 5 artists who generated the highest revenue from track sales


In [ ]:
sql = '''
SELECT ar.name AS artist,
       ROUND(SUM(il.unit_price * il.quantity), 2) AS revenue
FROM artist ar
JOIN album al ON al.artist_id = ar.artist_id
JOIN track t ON t.album_id = al.album_id
JOIN invoice_line il ON il.track_id = t.track_id
GROUP BY ar.artist_id, ar.name
ORDER BY revenue DESC
LIMIT 5;
'''
df4 = pd.read_sql_query(sql, conn)
display(df4)

## Q5. Countries by average revenue per invoice


In [ ]:
sql = '''
SELECT billing_country AS country,
       COUNT(invoice_id) AS invoice_count,
       ROUND(AVG(total), 2) AS average_revenue_per_invoice
FROM invoice
GROUP BY billing_country
ORDER BY average_revenue_per_invoice DESC;
'''
df5 = pd.read_sql_query(sql, conn)
display(df5)

## Q6. Media type with number of tracks


In [ ]:
sql = '''
SELECT m.name AS media_type, COUNT(t.track_id) AS track_count
FROM media_type m
JOIN track t ON t.media_type_id = m.media_type_id
GROUP BY m.media_type_id, m.name
ORDER BY track_count DESC;
'''
df6 = pd.read_sql_query(sql, conn)
display(df6)

## Q7. Artists who have released tracks in more than one genre


In [ ]:
sql = '''
SELECT ar.name AS artist, COUNT(DISTINCT t.genre_id) AS genre_count,
       GROUP_CONCAT(DISTINCT g.name) AS genres
FROM artist ar
JOIN album al ON al.artist_id = ar.artist_id
JOIN track t ON t.album_id = al.album_id
JOIN genre g ON g.genre_id = t.genre_id
GROUP BY ar.artist_id, ar.name
HAVING COUNT(DISTINCT t.genre_id) > 1
ORDER BY genre_count DESC, ar.name;
'''
df7 = pd.read_sql_query(sql, conn)
display(df7)

## Q8. Top 10 longest tracks by duration


In [ ]:
sql = '''
SELECT name AS track, milliseconds,
       ROUND(milliseconds / 60000.0, 2) AS duration_minutes
FROM track
ORDER BY milliseconds DESC
LIMIT 10;
'''
df8 = pd.read_sql_query(sql, conn)
display(df8)

## Q9. Top 5 artists' share of total revenue (Pie Chart)


In [ ]:
sql = '''
WITH artist_revenue AS (
    SELECT ar.artist_id, ar.name AS artist,
           SUM(il.unit_price * il.quantity) AS revenue
    FROM artist ar
    JOIN album al ON al.artist_id = ar.artist_id
    JOIN track t ON t.album_id = al.album_id
    JOIN invoice_line il ON il.track_id = t.track_id
    GROUP BY ar.artist_id, ar.name
),
top_artists AS (
    SELECT artist, revenue
    FROM artist_revenue
    ORDER BY revenue DESC
    LIMIT 5
)
SELECT artist, ROUND(revenue, 2) AS revenue,
       ROUND(100.0 * revenue / (SELECT SUM(revenue) FROM artist_revenue), 2) AS total_revenue_share_percent
FROM top_artists
ORDER BY revenue DESC;
'''
df9 = pd.read_sql_query(sql, conn)
display(df9)
df9.set_index('artist')['revenue'].plot(kind='pie', autopct='%1.1f%%', ylabel='', title='Revenue Distribution Among Top 5 Artists')
plt.tight_layout()
plt.show()

## Q10. Revenue by genre (Rock, Latin, Metal) in each country (Stacked Column Chart)


In [ ]:
sql = '''
SELECT i.billing_country AS country, g.name AS genre,
       ROUND(SUM(il.unit_price * il.quantity), 2) AS revenue
FROM invoice i
JOIN invoice_line il ON il.invoice_id = i.invoice_id
JOIN track t ON t.track_id = il.track_id
JOIN genre g ON g.genre_id = t.genre_id
WHERE g.name IN ('Rock', 'Latin', 'Metal')
GROUP BY i.billing_country, g.name
ORDER BY i.billing_country, g.name;
'''
df10 = pd.read_sql_query(sql, conn)
display(df10)
pivot10 = df10.pivot(index='country', columns='genre', values='revenue').fillna(0)
pivot10.plot(kind='bar', stacked=True, figsize=(12, 6), title='Revenue by Genre and Country')
plt.xlabel('Country')
plt.ylabel('Revenue')
plt.xticks(rotation=60, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
conn.close()
